In [2]:
# =======================================================
# STEP 7: Convert Audio to Normalized Grayscale Mel-Spectrogram Arrays
# =======================================================
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for faster processing
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import os # Import the os module
import librosa # Import the librosa module
import numpy as np # Import the numpy module
from scipy import ndimage # Import ndimage from scipy


# Parameters
AUGMENTED_AUDIO_PATH = "/content/drive/MyDrive/Colab Notebooks/Main Project/augmented_audio"
SPECTROGRAM_ARRAY_PATH = "/content/drive/MyDrive/Colab Notebooks/Main Project/spectrogram_arrays"
os.makedirs(SPECTROGRAM_ARRAY_PATH, exist_ok=True)

# Mel-spectrogram parameters optimized for MobileNetV2
SAMPLE_RATE = 22050
DURATION = 5.0
N_MELS = 224  # Matches MobileNetV2 input height
HOP_LENGTH = 512
N_FFT = 2048
FMAX = 8000   # Focus on most relevant frequencies for environmental sounds

def audio_to_mel_array(audio_path, save_path):
    """
    Convert audio file to normalized grayscale Mel-spectrogram and save as .npy array
    Returns the array shape for verification
    """
    try:
        # Load and ensure mono audio
        y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

        # Create Mel-spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=N_MELS,
            hop_length=HOP_LENGTH,
            n_fft=N_FFT,
            fmax=FMAX,
            power=2.0  # Use power spectrogram for better dynamic range
        )

        # Convert to log scale (decibels)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalize to [0, 1] range for grayscale
        mel_normalized = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-8)

        # Ensure correct shape (224, 224) for MobileNetV2
        target_shape = (224, 224)
        if mel_normalized.shape != target_shape:
            # Resize using interpolation if shape doesn't match

            zoom_factors = (target_shape[0] / mel_normalized.shape[0],
                          target_shape[1] / mel_normalized.shape[1])
            mel_normalized = ndimage.zoom(mel_normalized, zoom_factors)

        # Save as .npy file
        np.save(save_path, mel_normalized.astype(np.float32))

        return mel_normalized.shape, True

    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None, False

# Process all audio files
print("Converting augmented audio to normalized Mel-spectrogram arrays...")

success_count = 0
total_count = 0
shape_info = []

for class_folder in tqdm(os.listdir(AUGMENTED_AUDIO_PATH)):
    class_audio_path = os.path.join(AUGMENTED_AUDIO_PATH, class_folder)
    class_array_path = os.path.join(SPECTROGRAM_ARRAY_PATH, class_folder)
    os.makedirs(class_array_path, exist_ok=True)

    for audio_file in os.listdir(class_audio_path):
        if audio_file.endswith('.wav'):
            audio_path = os.path.join(class_audio_path, audio_file)
            array_filename = audio_file.replace('.wav', '.npy')
            array_path = os.path.join(class_array_path, array_filename)

            shape, success = audio_to_mel_array(audio_path, array_path)
            if success:
                success_count += 1
                shape_info.append(shape)
            total_count += 1

print(f"✅ Array conversion complete!")
print(f"Successfully processed: {success_count}/{total_count} files")
print(f"Output shape: {shape_info[0] if shape_info else 'No files processed'}")
print(f"Data type: float32")
print(f"Value range: [0, 1] (normalized)")

Converting augmented audio to normalized Mel-spectrogram arrays...


100%|██████████| 13/13 [10:50<00:00, 50.03s/it]

✅ Array conversion complete!
Successfully processed: 3120/3120 files
Output shape: (224, 224)
Data type: float32
Value range: [0, 1] (normalized)
